# Assignment_9_CCTV_Object_Detection

Apply object detection from CCTV footage.


## Aim
Apply object detection to CCTV footage using YOLO.

**Input:** your own `.mp4`, `.avi`, or `.mov` CCTV/video file.

**Output:** annotated video with detected objects.

The result ZIP contains the processed video and a detection summary.


In [ ]:
# Install only the object-detection package. Do not upgrade NumPy/Pandas.
!pip -q install ultralytics --no-deps


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.2 MB/s eta 0:00:00


In [ ]:
from google.colab import files
import os, shutil, json
from collections import Counter

uploaded = files.upload()
VIDEO_PATH = next(iter(uploaded.keys()))
RESULT_DIR = "/content/results"
os.makedirs(RESULT_DIR, exist_ok=True)
print("Video:", VIDEO_PATH)


Saving WhatsApp Video 2026-10-06 at 00.27.03.mp4 to WhatsApp Video 2026-10-06 at 00.27.03 (1).mp4
Video: WhatsApp Video 2026-10-06 at 00.27.03 (1).mp4


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.predict(
    source=VIDEO_PATH,
    save=True,
    project="/content",
    name="cctv_detected",
    conf=0.35,
    verbose=True
)

OUTPUT_DIR="/content/cctv_detected"
print("Processed output folder:", OUTPUT_DIR)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/452) /content/WhatsApp Video 2026-10-06 at 00.27.03 (1).mp4: 384x640 3 persons, 4 cars, 461.8ms
video 1/1 (frame 2/452) /content/WhatsApp Video 202

In [ ]:
# ==========================================
# ASSIGNMENT 9 - FINAL RESULT & DOWNLOAD
# ==========================================

import os
import glob
import shutil
import pandas as pd
from collections import Counter
from google.colab import files

# Result folder
RESULT_DIR = "/content/results"
os.makedirs(RESULT_DIR, exist_ok=True)

# Count detected objects
counts = Counter()

for r in results:
    if r.boxes is not None and r.boxes.cls is not None:
        for cls_id in r.boxes.cls.cpu().numpy().astype(int):
            counts[model.names[cls_id]] += 1

# Create result table
summary = pd.DataFrame(
    [
        {"Object": obj, "Detections": count}
        for obj, count in counts.most_common()
    ]
)

print("========== OBJECT DETECTION RESULTS ==========")
display(summary)

# Save result table
csv_path = f"{RESULT_DIR}/detection_summary.csv"
summary.to_csv(csv_path, index=False)

# Find processed video
video_files = glob.glob("/content/cctv_detected/*")

for video in video_files:
    if video.lower().endswith((".mp4", ".avi", ".mov")):
        shutil.copy2(video, RESULT_DIR)
        print("Processed video copied:", video)
        break

# Create ZIP
zip_path = shutil.make_archive(
    "/content/Assignment_9_Results",
    "zip",
    RESULT_DIR
)

print("\n==========================================")
print("RESULTS CREATED SUCCESSFULLY")
print("==========================================")
print("CSV:", csv_path)
print("ZIP:", zip_path)

# Download ZIP
files.download(zip_path)

========== OBJECT DETECTION RESULTS ==========


,Object,Detections
0,car,2434
1,truck,234
2,bus,59
3,person,51
4,train,2


Processed video copied: /content/cctv_detected/WhatsApp Video 2026-10-06 at 00.27.03 (1).avi

RESULTS CREATED SUCCESSFULLY
CSV: /content/results/detection_summary.csv
ZIP: /content/Assignment_9_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Final Result + Download Cell
# This cell is self-contained so it will not fail with NameError for pd/shutil/etc.

import os
import glob
import shutil
import pandas as pd
from collections import Counter

RESULT_DIR = "/content/results"
os.makedirs(RESULT_DIR, exist_ok=True)

# Make sure the YOLO model and detection results exist.
try:
    model
    results
except NameError:
    from ultralytics import YOLO
    if "VIDEO_PATH" not in globals():
        raise RuntimeError(
            "VIDEO_PATH is not defined. Run the video-upload cell first."
        )
    model = YOLO("yolo11n.pt")
    results = model.predict(
        source=VIDEO_PATH,
        save=True,
        project="/content",
        name="cctv_detected",
        conf=0.35,
        verbose=True
    )

# Count detected objects.
counts = Counter()

for r in results:
    if r.boxes is not None and r.boxes.cls is not None:
        for cls_id in r.boxes.cls.cpu().numpy().astype(int):
            counts[model.names[cls_id]] += 1

summary = pd.DataFrame(
    [{"Object": k, "Detections": v} for k, v in counts.most_common()],
    columns=["Object", "Detections"]
)

print("Object Detection Result")
display(summary)

summary.to_csv(f"{RESULT_DIR}/detection_summary.csv", index=False)

# Copy processed video to result folder.
output_dir = "/content/cctv_detected"
videos = glob.glob(output_dir + "/*")

copied_video = False
for v in videos:
    if v.lower().endswith((".mp4", ".avi", ".mov")):
        shutil.copy2(v, RESULT_DIR)
        print("Processed video copied:", os.path.basename(v))
        copied_video = True
        break

if not copied_video:
    print("Processed video was not found in /content/cctv_detected.")
    print("Check the folder in the Colab Files panel.")

# Create ZIP.
zip_file = shutil.make_archive(
    "/content/Assignment_9_Results",
    "zip",
    RESULT_DIR
)

print("Created:", zip_file)

# Download ZIP.
from google.colab import files
files.download(zip_file)


Object Detection Result


,Object,Detections
0,car,2434
1,truck,234
2,bus,59
3,person,51
4,train,2


Processed video copied: WhatsApp Video 2026-10-06 at 00.27.03 (1).avi
Created: /content/Assignment_9_Results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>